# Лабораторна робота №7 (Варіант 2)
## Інтегровані пайплайни та оркестрація за допомогою агента LlamaIndex

**Дисципліна:** Технології та системи нейронних мереж  
**Університет:** Національний університет «Львівська політехніка»  
**Тема:** Автономна інтеграція інструментів студії (LlamaIndex Agent Orchestration & Persistent Gallery)  

### Мета роботи
Дослідити наскрізні інтегровані сценарії функціонування Мультимодальної Студії. Об'єднати візуальний аналіз LLaVA, креативну генерацію Stable Diffusion та базу даних галереї під керуванням інтелектуального агента `LlamaIndex AgentWorkflow`. Зафіксувати стрімінгові трейси та продемонструвати персистентність результатів у SQLite.

In [1]:
import sys
import json
import asyncio
from pathlib import Path
import pandas as pd

def resolve_lab_dir() -> Path:
    current = Path.cwd().resolve()
    if (current / 'src').exists() and (current / 'assets').exists():
        return current
    if (current / 'lab7_multimodal_studio').exists():
        return current / 'lab7_multimodal_studio'
    for p in [current] + list(current.parents):
        cand = p / 'lab7_multimodal_studio'
        if cand.exists():
            return cand
    return current

LAB_DIR = resolve_lab_dir()
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from src.storage.gallery_service import GalleryService
from src.agent.studio_agent import StudioAgent

gallery = GalleryService()
records = gallery.get_all_records(limit=10)
print(f'Зафіксовано записів у галереї: {len(records)}')
for r in records[:5]:
    print(f"- [{r['id']}] task={r['task_type']} | latency={r['latency_sec']}s | prompt={str(r['prompt'])[:40]}")

C:\Users\Eclipse\PycharmProjects\neural-network-technologies-and-systems\.venv-lab04\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Зафіксовано записів у галереї: 6
- [caption_20261111_113935_011] task=caption | latency=0.682s | prompt=concise
- [transformation_20261111_113909_697] task=transformation | latency=4.933s | prompt=Cyberpunk neon city street at night, glo
- [transformation_20261111_113904_699] task=transformation | latency=6.13s | prompt=An oil painting in the style of Vincent 
- [vqa_20261111_113846_777] task=vqa | latency=0.387s | prompt=What electronic test equipment is visibl
- [caption_20261111_113846_322] task=caption | latency=3.101s | prompt=concise


### 1. Автономне виконання багатокрокового запиту агентом Студії

Задамо агенту комплексний запит: проаналізувати вміст робочого місця робототехніки, скласти лаконічний підпис та зробити вибірку попередніх операцій з бази даних галереї.

In [2]:
agent = StudioAgent(gallery_service=gallery)
robotics_img_path = str(LAB_DIR / 'assets' / '04_modern_robotics_lab.png')

prompt = (
    f"Inspect the image at '{robotics_img_path}'. "
    "Provide a concise caption describing its contents, and then inspect the studio gallery records."
)
print(f'User Prompt: {prompt}\n')

response, traces, elapsed = await agent.run_request(prompt)
print(f'Час виконання запиту: {elapsed:.2f}с\n')
print(f'Синтезована відповідь агента:\n{response}\n')
print(f'Кількість зафіксованих подій у трейсі: {len(traces)}')
for tr in traces:
    print(f"  - Event: {tr['event']}, Tool: {tr.get('tool')}")

User Prompt: Inspect the image at 'C:\Users\Eclipse\PycharmProjects\neural-network-technologies-and-systems\max\lab7_multimodal_studio\assets\04_modern_robotics_lab.png'. Provide a concise caption describing its contents, and then inspect the studio gallery records.



Loading weights:   0%|          | 0/715 [00:00<?, ?it/s]

Loading weights:   6%|▋         | 45/715 [00:00<00:03, 221.25it/s]

Loading weights:  10%|█         | 73/715 [00:00<00:02, 240.54it/s]

Loading weights:  14%|█▎        | 98/715 [00:00<00:02, 229.95it/s]

Loading weights:  18%|█▊        | 129/715 [00:00<00:02, 250.28it/s]

... (16 outputs left out)

Loading weights:  94%|█████████▎| 670/715 [00:02<00:00, 298.46it/s]

Loading weights:  98%|█████████▊| 701/715 [00:02<00:00, 276.04it/s]

Loading weights: 100%|██████████| 715/715 [00:02<00:00, 274.36it/s]

Час виконання запиту: 32.63с

Синтезована відповідь агента:
The image depicts "A robot with a large screen and a small robot."

Recent studio gallery records include:
- A concise caption task (ID: caption_20261111_114348_215) completed in 2.949 seconds.
- Another concise caption task (ID: caption_20261111_113935_011) completed in 0.682 seconds.
- A cyberpunk neon city street transformation (ID: transformation_20261111_113909_697) with a 4.933-second latency.
- An oil painting in the style of Van Gogh (ID: transformation_20261111_113904_699) with a 6.13-second latency.
- A visual question answer about electronic test equipment on a workbench (ID: vqa_20261111_113846_777) with a 0.387-second latency.

Кількість зафіксованих подій у трейсі: 4
  - Event: ToolCall, Tool: caption_image
  - Event: ToolCallResult, Tool: caption_image
  - Event: ToolCall, Tool: inspect_studio_gallery
  - Event: ToolCallResult, Tool: inspect_studio_gallery


### 2. Аудит збережених артефактів у Persistent Gallery

Перевіримо оновлений стан персистентної галереї після виконання операцій агентом:

In [3]:
updated_records = gallery.get_all_records(limit=20)
df_audit = pd.DataFrame([
    {
        'id': r['id'],
        'task': r['task_type'],
        'prompt': (r['prompt'] or '')[:45],
        'latency_s': r['latency_sec'],
        'has_input_img': bool(r.get('input_image_path')),
        'has_output_img': bool(r.get('output_image_path'))
    }
    for r in updated_records
])
print('=== Зведена таблиця записів у Persistent Gallery (SQLite) ===')
display(df_audit)

=== Зведена таблиця записів у Persistent Gallery (SQLite) ===


,id,task,prompt,latency_s,has_input_img,has_output_img
0,caption_20261111_114348_215,caption,concise,2.949,True,False
1,caption_20261111_113935_011,caption,concise,0.682,True,False
2,transformation_20261111_113909_697,transformation,"Cyberpunk neon city street at night, glowing",4.933,True,True
3,transformation_20261111_113904_699,transformation,An oil painting in the style of Vincent van G,6.130,True,True
4,vqa_20261111_113846_777,vqa,What electronic test equipment is visible on,0.387,True,False
5,caption_20261111_113846_322,caption,concise,3.101,True,False
6,test_20261111_113747_950,test,unit test record,0.050,True,False


### 3. Висновки до блокноту 2

- Створено наскрізний агентний конвеєр на основі `LlamaIndex`, що динамічно обирає та викликає спеціалізовані інструменти студії.
- Підтверджено надійність інтеграції пам'яті та структурних трейсів: кожен виклик (`caption_image`, `inspect_studio_gallery`) фіксується з параметрами та вихідними даними.
- Доведено повну автономність збереження артефактів: результати зберігаються в локальній базі даних SQLite (`gallery.db`) та файловому сховищі `outputs/gallery/`, зберігаючи стан після перезапуску додатку.